# 12 - Evaluating the RAG patterns and the multi-agent graph

A small, **stratified sample** of the golden Q&A set (factual, table lookups, near-duplicates, version conflicts,
comparisons, unanswerable ...) is run through every pipeline and, optionally, the whole graph.

| Metric | Meaning |
|---|---|
| `hit_rate` | an expected source document is among the retrieved chunks |
| `mrr` | 1 / rank of the first chunk from an expected document |
| `golden_match` | key facts of the golden answer (numbers, codes, most words) appear in the answer |
| `judge_mean`, `judge_pass_rate` | judge LLM score and share at or above `JUDGE_PASS_SCORE` |
| `latency_mean_s` | seconds per question |

The same evaluation runs from the shell: `make evaluate SAMPLE=10` (add `GRAPH=1` for the graph).

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
from agentic_rag.evaluation.batch import load_golden, evaluate_pipelines, evaluate_graph, summarize, save_report

items = load_golden(settings, sample=8, seed=11)
for i in items:
    print(f"{i.id} {i.category:<15} {i.question_type:<17} {i.question[:70]}")

## Pipelines (hybrid, advanced, multi-query, corrective) with the judge

In [ ]:
rows = evaluate_pipelines(settings, items, use_judge=True, progress=print)

In [ ]:
def show(summary):
    cols = ["n", "errors", "hit_rate", "mrr", "golden_match", "judge_mean", "judge_pass_rate", "latency_mean_s"]
    print(f"{'pipeline':<12}" + "".join(f"{c:>16}" for c in cols))
    for name, values in summary.items():
        print(f"{name:<12}" + "".join(f"{'-' if values[c] is None else values[c]:>16}" for c in cols))

show(summarize(rows))

## Where did the pipelines disagree?

In [ ]:
from collections import defaultdict

by_item = defaultdict(dict)
for r in rows:
    by_item[r["id"]][r["pipeline"]] = r
for item in items:
    results = by_item[item.id]
    marks = "  ".join(f"{p}:{'Y' if r.get('golden_match') else '.'}" for p, r in results.items())
    print(f"{item.id} {item.question_type:<17} {marks}")

## The full multi-agent graph on a few questions
Approval interrupts are auto-rejected and clarifications get a fixed reply, so the batch runs unattended.

In [ ]:
from agentic_rag.graph import CoordinatorService

async with CoordinatorService.open(settings) as service:
    graph_rows = await evaluate_graph(service, items[:3], progress=print)
show(summarize(rows + graph_rows))

In [ ]:
path = save_report(settings, "notebook-eval", items, rows + graph_rows, summarize(rows + graph_rows))
print("Saved", path)

## What to observe
- Hybrid is fastest; advanced and corrective trade latency for precision and honesty.
- Version-conflict and near-duplicate questions separate the pipelines most clearly.
- The graph adds planning and judging overhead - worth it for multi-domain questions, not for simple lookups.